# Datamine MINWID Process Example

This notebook demonstrates how to configure and run the **`minwid`** process wrapper in `dmstudio`.

## Process Description

## MINWID

Composite drillhole data into ore and waste intervals that satisfy the specified mining width criteria.

**Note** : To composite drillhole data by optimizing the composite interval using ore and waste criteria, see [COMPSE Process](<compse.md>).

The input file must be in standard sample format (as output by process **[HOLES3D](<holes3d.md>)**). The input file must be in the order of **BHID** and **FROM** , that is, sorted in drillhole order in increasing downhole distance.

This is the order output from the **HOLES3D** process. The output file is in standard drillhole format supplemented with the **VALUE** and **DENSITY** fields. The initial classification of ore and waste is based on whether a specified numeric field in the input file is above or below a specified cutoff.

The contiguous intervals of like samples make up either 'narrow' or 'wide' ore, or 'narrow' or 'wide' waste. If the ore interval is greater than the minimum ore width, then the ore is 'wide'. If the waste is greater than the minimum waste width, then the waste is 'wide'. 'Narrow' ore and waste intervals must be recombined to satisfy the mining width criteria. If 'narrow' ore cannot be recombined with other ore intervals, the option to dilute with adjacent waste is allowed. To allow minimum dilution of 'narrow' ore intervals from adjacent waste a dilution splitting interval can be specified.

Although the ore body orientation is not explicit in the drill hole data, dilution can be added to hanging and/or foot wall contacts. A constant relationship between ore body and hole orientation is assumed, with the dilution width specified as a positive distance up and down the drill hole from an ore interval. An interval file is output to provide a convenient summary of ore intervals, dilution components,and additional material above cutoff that fails to meet the specified criteria. In additional to BHID, FROM and TO fields other category fields output to flag the interval status:

* **CATA** flags intervals that together satisfying all mining width criteria

* **CATB** flags intervals that fail the mining width criteria when wall dilution is added

* **CATC** flags above cutoff material that fails the minimum ore width criteria

* **CATD** flags the below cutoff material in mining intervals (1 - from DILP. 2 - from DILN, 3 -other internal waste)

### Input Files:

* **in** (Drillhole):
  Input sample file, in BHID and FROM order.
  Required=Yes

### Output Files:

* **out** (Drillhole):
  Output file of composites in standard
  Required=Yes

* **interval** (Undefined):
  Output file of composite interval and dilution types.
  Required=Yes

### Fields:

* **value** (Numeric : IN):
  Value which is to control compositing. This may be a grade or a calculated equivalent
  value from grades of different metals.
  Default=Undefined
  Required=Yes

* **density** (Numeric : IN):
  Density field.
  Default=Undefined
  Required=Yes

### Parameters:

* **cutoff**:
  Minimum value of **VALUE** which is worth mining (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **minore**:
  Minimum mining width for ore.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **minwaste**:
  Minimum width for internal waste (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **dilute**:
  Allow dilution of composites to create minimum ore width if non-zero (1).
  Range=0,1
  Values=0,1
  Default=1
  Required=Yes

* **narwast1**:
  Test for carrying narrow waste to be applied to either [1] or both [2] adjacent wide
  ores (1).
  Range=1,2
  Values=1,2
  Default=1
  Required=Yes

* **narwast2**:
  Allow narrow waste to be expanded into adjacent wide ore to meet the minimum waste
  width if non-zero (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=Yes

* **dilp**:
  Dilution interval added to the ore composite in the down hole direction (0).
  Range=0,+
  Values=Undefined
  Default=0
  Required=Yes

* **diln**:
  Dilution interval added to the ore composite in the up hole direction (0).
  Range=0,+
  Values=Undefined
  Default=0
  Required=Yes

* **dilint**:
  Dilution splitting interval to be used when diluting narrow ore with adjacent waste
  (0).
  Range=0,+
  Values=Undefined
  Default=0
  Required=Yes

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('minwid')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute minwid
print("Running minwid...")
dm_cmd.minwid(
    in_i='t_assays',  # required
    out_o='t_minwid_out',  # required
    interval_o='t_minwid_out',  # required
    value_f='optional',  # required
    density_f='optional',  # required
    cutoff_p='required_val',  # required
    minore_p='required_val',  # required
    minwaste_p='required_val',  # required
    dilute_p='required_val',  # required
    narwasts_f=['optional'],  # required
    dilp_p='required_val',  # required
    diln_p='required_val',  # required
    dilint_p='required_val',  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("minwid execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_minwid_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")